# **Employee Data Profiling Overview**

This notebook performs data profiling on the Employee dataset to evaluate structure, quality, and consistency before further processing.

## Objectives

* Validate schema (columns, data types)
* Check missing values and duplicates
* Analyze data distribution (age, salary, etc.)
* Detect anomalies and inconsistencies

## Scope

* Row & column count
* Data types validation
* Null value analysis
* Distinct count (cardinality)
* Summary stats (min, max, mean, stddev)
* Basic outlier checks

## Outcome

Provides insights into data quality and readiness for cleaning, transformation, and analysis.

**Status:** Data profiling initiated...
**Author:** Ritik
**Env:** Development / Testing


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

print("Required libraries imported successfully. :) ")

Required libraries imported successfully. :) 


In [ ]:
spark = (
    SparkSession.builder
    .appName("EmployeesDataProfiling")
    .getOrCreate()
)

print("SparkSession Build Successfully :) ")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/22 10:44:17 WARN Utils: Your hostname, codespaces-5c0be5, resolves to a loopback address: 127.0.0.1; using 10.0.1.51 instead (on interface eth0)
26/04/22 10:44:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/22 10:44:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession Build Successfully :) 


In [ ]:
try :
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv("/workspaces/Docker/row_data/raw_employees.csv")
    )
    print(f"Data Loaded successfully row_count {df.count()}  :) ")

except Exception as e :
    print(f"Error Occurred : {e}")

Data Loaded successfully row_count 100  :) 


In [ ]:
df.sample(fraction=0.1, withReplacement=False).show(truncate=False)

+-----------+----------+-----------+-----------------+--------------------------------+--------------+----------------------+----------------+--------+---------------------+------------+------------------+--------------+-----------------+-------------------+---------+------------------+----------+
|employee_id|first_name|last_name  |full_name        |email                           |phone         |job_title             |department      |store_id|store_name           |store_city  |hire_date         |years_employed|annual_salary_usd|commission_rate_pct|is_active|performance_rating|manager_id|
+-----------+----------+-----------+-----------------+--------------------------------+--------------+----------------------+----------------+--------+---------------------+------------+------------------+--------------+-----------------+-------------------+---------+------------------+----------+
|4008       |raymond   |Campbell   |Raymond Campbell |raymond_campbell43@hotmail.com  |8217757314    |C

## **Employee ID data profiling**

In [ ]:
# Null check
df.select("employee_id").filter(F.col("employee_id").isNull()).count()

0

In [ ]:
# Uniqueness (duplicate)
df.groupBy("employee_id").count().filter("count > 1").show()

+-----------+-----+
|employee_id|count|
+-----------+-----+
+-----------+-----+



In [ ]:
# Data type validation
df.select("employee_id").dtypes

[('employee_id', 'int')]

In [ ]:
# Range check
df.select("employee_id").filter(F.col("employee_id") < 1).show()

+-----------+
|employee_id|
+-----------+
+-----------+



In [ ]:
df.select("employee_id").sample(fraction=0.1, withReplacement=False).show()

+-----------+
|employee_id|
+-----------+
|       4004|
|       4006|
|       4026|
|       4029|
|       4033|
|       4036|
|       4038|
|       4050|
|       4055|
|       4064|
|       4070|
|       4073|
|       4077|
|       4084|
|       4089|
|       4092|
|       4093|
+-----------+



## **Manager ID data Profiling**

In [ ]:

df.select("first_name","manager_id").filter(F.col("manager_id").isNull()).show()

+----------+----------+
|first_name|manager_id|
+----------+----------+
|    George|      NULL|
|     Emily|      NULL|
|     Laura|      NULL|
|     James|      NULL|
|     Frank|      NULL|
|   Joshua |      NULL|
|     tyler|      NULL|
|  nicholas|      NULL|
|     Jacob|      NULL|
|   Deborah|      NULL|
|   Deborah|      NULL|
|     Joyce|      NULL|
|     Frank|      NULL|
|      Mark|      NULL|
|     Laura|      NULL|
|   CAROLYN|      NULL|
|     Diane|      NULL|
+----------+----------+



In [ ]:
df.select("manager_id").filter(F.col("manager_id").isNull()).count()

17

In [ ]:
df.select("manager_id").filter(F.col("manager_id").isNotNull()).count()

83

In [ ]:
df.groupBy("manager_id").count().filter(F.col("manager_id").isNotNull()).filter("count > 1").show()

+----------+-----+
|manager_id|count|
+----------+-----+
|      4092|    2|
|      4061|    2|
|      4040|    2|
|      4032|    2|
|      4012|    2|
|      4037|    2|
|      4007|    2|
|      4023|    2|
|      4068|    2|
|      4020|    3|
|      4066|    2|
|      4010|    2|
|      4086|    2|
|      4055|    2|
|      4053|    2|
|      4098|    2|
|      4050|    2|
|      4003|    2|
|      4096|    2|
|      4021|    2|
+----------+-----+
only showing top 20 rows


In [ ]:
df.select("manager_id").dtypes

[('manager_id', 'int')]

## **performance_rating Data Profiling**

In [ ]:
df.select("performance_rating").dtypes

[('performance_rating', 'string')]

In [ ]:
df.select("performance_rating").distinct().show()

+------------------+
|performance_rating|
+------------------+
|         Excellent|
|                 3|
|                 5|
|                 B|
|           Average|
|     Below Average|
|              Good|
|                 D|
|                 C|
|                 A|
|                 4|
|                 2|
|              NULL|
+------------------+



In [ ]:
df.select("performance_rating").filter(F.col("performance_rating").isNotNull()).count()

94

In [ ]:
df.groupBy("performance_rating").count().show()

+------------------+-----+
|performance_rating|count|
+------------------+-----+
|         Excellent|   13|
|                 3|    9|
|              NULL|    6|
|                 5|    7|
|                 B|    4|
|           Average|    4|
|     Below Average|    1|
|              Good|   10|
|                 D|   12|
|                 C|   11|
|                 A|    7|
|                 4|    4|
|                 2|   12|
+------------------+-----+



## **Employee Is active or not data profiling**

In [ ]:
df.select("is_active").dtypes

[('is_active', 'string')]

In [ ]:
df.select("is_active").filter(F.col("is_active").isNull()).count()

0

In [ ]:
df.select("is_active").distinct().show()

+----------+
| is_active|
+----------+
|         Y|
|    Active|
|         N|
|Terminated|
|        No|
|       Yes|
|         1|
+----------+



In [ ]:
df.groupBy("is_active").count().show()

+----------+-----+
| is_active|count|
+----------+-----+
|         Y|   19|
|    Active|   19|
|         N|   14|
|Terminated|   12|
|        No|   10|
|       Yes|   13|
|         1|   13|
+----------+-----+



## **Commission_rate_pct Data Profiling**

In [ ]:
df.select("commission_rate_pct").dtypes

[('commission_rate_pct', 'double')]

In [ ]:
df.select("commission_rate_pct").filter(F.col("commission_rate_pct").isNull()).count()

0

In [ ]:
df.sample(fraction=0.1, withReplacement=False).select("commission_rate_pct").show()

+-------------------+
|commission_rate_pct|
+-------------------+
|                7.8|
|                6.4|
|                5.0|
|                1.3|
|                2.6|
|                5.2|
|                6.5|
|                4.6|
|                1.3|
|                2.3|
|                4.1|
|                5.2|
|                6.0|
|                1.2|
|                6.3|
|                7.4|
+-------------------+

